# 39 — Checkpoint & Resume

LangGraph's `SqliteSaver` checkpointer persists the full graph state to SQLite after every node. Resume a run from any `thread_id` — even across Python sessions — without re-executing completed steps.

**What you'll learn**
- `SqliteSaver.from_conn_string(path)` — SQLite-backed checkpointer
- `graph.compile(checkpointer=saver)` — attach saver; every node completion = checkpoint
- `config = {"configurable": {"thread_id": "..."}}` — thread-scoped state isolation
- `app.get_state(config)` — read persisted state without re-running
- `app.get_state_history(config)` — list all checkpoints for a thread

**Contrast:** `InMemoryStore` (example 36) persists user facts across threads but resets on process restart. `SqliteSaver` persists the entire graph execution state to disk.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langgraph python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. How SqliteSaver works ──────────────────────────────────────────────────
#
# Without checkpointer:             With SqliteSaver:
#   invoke(state, config)             invoke(state, config={thread_id: "t1"})
#   → runs all nodes                    step 1 → state saved to SQLite
#   → state gone on return              step 2 → state saved to SQLite
#   → no resume possible                step 3 → state saved to SQLite (done)
#
#                                     invoke({}, config={thread_id: "t1"})
#                                     → resumes from last checkpoint
#                                     → skips completed steps!
#
# thread_id scoping:
#   thread-0 and thread-1 have independent state
#   Both can be in-flight simultaneously
#   Both can be resumed independently

DB_PATH = "/tmp/checkpoint_demo.sqlite"
import os
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
print(f"Checkpoint DB: {DB_PATH}")
print("Each invoke() with the same thread_id resumes from the last checkpoint.")

In [ ]:
# ── 4. Build the checkpointed graph ──────────────────────────────────────────
from typing import TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, StateGraph

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

STEP_PROMPTS = [
    "Briefly introduce the topic in 1-2 sentences: {task}",
    "List 3 key points about the topic: {task}",
    "Write a concise conclusion (1-2 sentences): {task}",
]


class CheckpointState(TypedDict):
    task: str
    step: int
    outputs: list[str]
    done: bool


def step_node(state: CheckpointState) -> dict:
    step = state["step"]
    if step >= len(STEP_PROMPTS):
        return {"done": True}
    prompt = STEP_PROMPTS[step].format(task=state["task"])
    result = llm.invoke([HumanMessage(content=prompt)])
    print(f"  Step {step + 1}/{len(STEP_PROMPTS)} complete")
    return {
        "outputs": state["outputs"] + [result.content],
        "step": step + 1,
        "done": step + 1 >= len(STEP_PROMPTS),
    }


def should_continue(state: CheckpointState) -> str:
    return END if state["done"] else "step"


graph = StateGraph(CheckpointState)
graph.add_node("step", step_node)
graph.set_entry_point("step")
graph.add_conditional_edges("step", should_continue, {"step": "step", END: END})

saver = SqliteSaver.from_conn_string(DB_PATH)
app = graph.compile(checkpointer=saver)
print("Graph compiled with SqliteSaver checkpointer.")

In [ ]:
# ── 5. Run, inspect checkpoints, and resume ───────────────────────────────────
TASK = "Explain the three laws of robotics."
THREAD_ID = "thread-0"
config = {"configurable": {"thread_id": THREAD_ID}}

print(f"TASK: {TASK}\n")
result = app.invoke({"task": TASK, "step": 0, "outputs": [], "done": False}, config=config)
print(f"\nSteps completed: {len(result['outputs'])}")
print(f"Final output preview: {result['outputs'][-1][:150]}")

# Inspect persisted state
print("\n--- Checkpoint inspection ---")
snapshot = app.get_state(config)
print(f"Persisted step: {snapshot.values['step']}")
print(f"Persisted done: {snapshot.values['done']}")
print(f"Outputs count:  {len(snapshot.values['outputs'])}")

# List all checkpoints
history = list(app.get_state_history(config))
print(f"\nTotal checkpoints saved: {len(history)}")
for i, h in enumerate(history):
    print(f"  Checkpoint {i}: step={h.values.get('step')}, done={h.values.get('done')}")

# Resume: same thread_id, no re-execution of completed steps
print("\n--- Resume (thread already done, no new LLM calls) ---")
resumed = app.invoke({}, config=config)
print(f"Resumed outputs count: {len(resumed['outputs'])} (unchanged)")

## Exercises

**Exercise 1 — Simulate interruption:** Add `raise Exception("interrupted!")` inside `step_node` after step 1. Run the cell — it fails. Remove the exception and re-run — does it resume from step 2?

**Exercise 2 — Parallel threads:** Run two different tasks with `thread-0` and `thread-1`. Call `app.get_state` on each — verify they're independent.

**Exercise 3 — Time-travel:** After a full run, call `app.get_state_history(config)` and select an intermediate checkpoint by `checkpoint_id`. Re-invoke from that point.

**Exercise 4 — MemorySaver vs SqliteSaver:** Replace `SqliteSaver` with `MemorySaver` (from `langgraph.checkpoint.memory`). What changes? When would you prefer each?

## What's next

- **11-hitl-approval** — combine checkpointing with interrupt/resume for human-in-the-loop
- **36-long-term-memory** — `InMemoryStore` for cross-thread user facts (vs. full state)
- **38-langgraph-command-pattern** — `Command` routing with checkpointing enabled